# 03. Experimentación y Selección de Modelos

Con los datos limpios, enriquecidos y escalados, es hora de encontrar el algoritmo predictivo ganador.

### Instrucciones Generales:
1. **Validación:** No entrenes y midas sobre el mismo conjunto (sobreajuste). Recuerda haber dividido en Entrenamiento y Prueba antes.
2. **Entrenamiento Base:** Entrena los siguientes modelos base con tu set de Entrenamiento y compáralos usando RMSE (Error Cuadrático Medio):
   - `LinearRegression`
   - `SGDRegressor`
   - `DecisionTreeRegressor`
   - `RandomForestRegressor`
3. **Cross Validation (Validación Cruzada):** Para tener una métrica robusta, usa `cross_val_score` en el set de Entrenamiento para cada uno de los modelos anteriores.
4. **Ajuste Fino (Fine Tuning):** Toma el modelo ganador y busca sus mejores hiperparámetros. Utiliza un `GridSearchCV` explorando el número de estimadores (`n_estimators`), las características máximas (`max_features`), etc.
5. **Conclusión y Benchmark (IMPORTANTE):** Redacta una conclusión comparando los algoritmos. Explica por qué escogiste el modelo final y valida tu decisión calculando el RMSE sobre tu Set de Prueba que habías reservado. Documenta si alguno de tus modelos se sobreajusto o subajusto. Recuerda que el modelo final no puede tener esos problemas!


In [11]:
# Empieza importando los algoritmos desde Scikit-Learn
from time import time

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from typing import Dict, Tuple, Any
from sklearn.model_selection import GridSearchCV, ParameterGrid
import joblib
import os
import sys
from IPython.display import display
import time

In [12]:
sys.path.append(os.path.abspath("../src/features")) 
# Si tu archivo está directamente en src, usa: sys.path.append(os.path.abspath("../src"))
from build_features import preprocess_pipeline
    
# Regenerar artefactos rápido si no existen en disco
df_train_raw = pd.read_csv('../data/interim/train_set.csv')
X_train_processed_dummy, artefactos_guardados = preprocess_pipeline(df_train_raw, is_train=True)
joblib.dump(artefactos_guardados, '../models/preprocessing_artifacts.pkl')
X_train_processed_dummy

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,rooms_per_household,bedrooms_per_room,population_per_household,rooms_per_population,dist_to_LA,dist_to_SF,ocean_INLAND,ocean_ISLAND,ocean_NEAR BAY,ocean_NEAR OCEAN
0,-1.425378,0.998764,1.891157,0.317134,1.352377,0.122984,1.386769,-1.005485,458300.0,-0.919586,2.135340,-1.641758,0.154018,1.167852,-1.532075,0,0,1,0
1,0.591519,-0.706077,0.932354,-0.302225,-0.440406,-0.702651,-0.377837,1.528191,483800.0,0.056315,-0.611182,-1.250924,0.930610,-1.038054,0.632102,0,0,0,0
2,-1.205716,1.259614,0.373052,-0.705083,-0.763154,-0.797010,-0.779239,-0.793143,101700.0,-0.015726,-0.254310,-0.217157,-0.011933,1.262291,-1.240750,1,0,0,0
3,1.225544,-0.887740,-0.905352,0.706746,0.730439,0.367045,0.724716,-0.901886,96100.0,-0.008784,-0.192318,-0.678656,0.296860,-0.657810,1.079725,1,0,0,0
4,0.706342,-0.878424,0.612753,0.794443,1.578537,0.427834,1.746468,-0.096857,361800.0,-0.701002,1.104943,-1.519000,0.316332,-0.970706,0.812670,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15734,0.451734,-0.687444,-0.266150,1.820314,0.744574,0.978560,0.888926,2.991395,419000.0,1.052145,-1.503329,0.034338,0.630737,-0.937027,0.552204,0,0,0,0
15735,0.916020,-0.747999,0.612753,-0.754412,-0.701903,-0.572909,-0.701044,-0.501023,118100.0,-0.505475,0.608209,0.588433,-0.676752,-0.910920,0.823578,1,0,0,0
15736,0.127233,0.309375,-0.425950,0.145394,0.120281,-0.019461,0.078302,-0.704027,88800.0,0.069730,-0.242848,-0.332999,0.124704,-0.009547,-0.263097,1,0,0,0
15737,1.250505,-1.428072,-1.224953,0.590730,0.551397,1.246211,0.672586,0.241313,148800.0,-0.091112,-0.301358,0.852367,-0.503751,-0.308300,1.476237,0,0,0,1


In [13]:
# 2. Separar características (X) y objetivo (y)
X_train_prepared = X_train_processed_dummy.drop("median_house_value", axis=1)
y_train = X_train_processed_dummy["median_house_value"].copy()

print(f"X_train_prepared shape: {X_train_prepared.shape}")
print(f"y_train shape: {y_train.shape}")

X_train_prepared shape: (15739, 18)
y_train shape: (15739,)


In [14]:
def evaluate_single_model(
    model: Any, 
    X_train: pd.DataFrame, 
    y_train: pd.Series, 
    cv_folds: int = 10
) -> Tuple[float, np.ndarray]:
    """
    Entrena y evalúa un modelo individual garantizando idempotencia.
    
    Args:
        model: Instancia del estimador de Scikit-Learn.
        X_train: Dataset de características preparadas.
        y_train: Variable objetivo.
        cv_folds: Número de particiones para la validación cruzada.
        
    Returns:
        Tupla con (RMSE_Entrenamiento, Array_RMSE_Validacion_Cruzada).
    """
    # Entrenamiento puro sobre todo el set
    model.fit(X_train, y_train)
    predicciones = model.predict(X_train)
    
    # SOLUCIÓN AL ERROR: Usamos np.sqrt en lugar del argumento 'squared=False'
    rmse_train = np.sqrt(mean_squared_error(y_train, predicciones))
    
    # Validación Cruzada
    scores_cv = cross_val_score(
        estimator=model, 
        X=X_train, 
        y=y_train, 
        scoring="neg_root_mean_squared_error", 
        cv=cv_folds,
        n_jobs=-1 # Utiliza todos los núcleos del procesador para mayor velocidad
    )
    
    # Scikit-Learn devuelve valores negativos para las métricas de error en CV, los invertimos
    rmse_cv_array = -scores_cv
    
    return rmse_train, rmse_cv_array


def run_base_experiments(X_train: pd.DataFrame, y_train: pd.Series) -> pd.DataFrame:
    """
    Orquesta el entrenamiento base de múltiples algoritmos y compila los resultados.
    Fija los random_states aquí para garantizar la idempotencia de los experimentos.
    """
    # Diccionario de modelos base. Fila de configuración centralizada.
    # Se instancian aquí para asegurar un estado limpio en cada ejecución.
    modelos = {
        "LinearRegression": LinearRegression(),
        "SGDRegressor": SGDRegressor(random_state=42),
        "DecisionTreeRegressor": DecisionTreeRegressor(random_state=42),
        "RandomForestRegressor": RandomForestRegressor(random_state=42, n_estimators=100)
    }
    
    lista_resultados = []
    
    print("Iniciando experimentación base y validación cruzada...")
    
    for nombre, modelo in modelos.items():
        print(f"Entrenando {nombre}...")
        rmse_train, rmse_cv_array = evaluate_single_model(modelo, X_train, y_train, cv_folds=10)
        
        rmse_cv_mean = rmse_cv_array.mean()
        rmse_cv_std = rmse_cv_array.std()
        
        # Lógica heurística para detectar sobreajuste (Overfitting)
        # Si el error en CV es un 20% mayor que el error en entrenamiento, levantamos bandera
        alerta_sobreajuste = "Sí ⚠️" if (rmse_cv_mean - rmse_train) > (rmse_train * 0.2) else "No ✅"
        
        lista_resultados.append({
            "Algoritmo": nombre,
            "RMSE Train": rmse_train,
            "RMSE CV (Promedio)": rmse_cv_mean,
            "RMSE CV (Desv. Estándar)": rmse_cv_std,
            "Alerta Sobreajuste": alerta_sobreajuste
        })
        
    # Convertimos los resultados a un DataFrame de Pandas para una visualización tabular elegante
    df_resultados = pd.DataFrame(lista_resultados)
    
    # Ordenamos del mejor modelo (menor RMSE promedio en CV) al peor
    df_resultados.sort_values(by="RMSE CV (Promedio)", ascending=True, inplace=True)
    df_resultados.reset_index(drop=True, inplace=True)
    
    print("¡Experimentación finalizada!")
    return df_resultados


df_benchmark = run_base_experiments(X_train_prepared, y_train)

display(df_benchmark)

Iniciando experimentación base y validación cruzada...
Entrenando LinearRegression...
Entrenando SGDRegressor...
Entrenando DecisionTreeRegressor...
Entrenando RandomForestRegressor...
¡Experimentación finalizada!


,Algoritmo,RMSE Train,RMSE CV (Promedio),RMSE CV (Desv. Estándar),Alerta Sobreajuste
0,RandomForestRegressor,16134.418999,43704.959848,1365.114023,Sí ⚠️
1,LinearRegression,57746.635202,57872.492811,1425.280353,No ✅
2,SGDRegressor,58178.822539,58182.884086,1442.009821,No ✅
3,DecisionTreeRegressor,0.000000,62118.446683,1927.544927,Sí ⚠️


In [15]:
def optimized_sgd_tuning(X_train: pd.DataFrame, y_train: pd.Series) -> SGDRegressor:
    """
    Ejecuta una búsqueda optimizada en cuadrícula basada en literatura 
    para encontrar los mejores hiperparámetros de un SGDRegressor.
    """
    print("Iniciando Búsqueda Optimizada para SGDRegressor...")
    
    # Instanciamos el modelo base
    sgd_base = SGDRegressor(random_state=42, max_iter=2000, early_stopping=True)
    
    # MALLA DE HIPERPARÁMETROS REDUCIDA (Basada en Benchmarks)
    param_grid = {
        'loss': ['squared_error', 'huber'],         # Descartamos las menos eficientes
        'penalty': ['l2', 'elasticnet'],            # l2 es Ridge (clásico), elasticnet es mixto
        'alpha': [0.0001, 0.001, 0.01],             # Valores clásicos de regularización
        'learning_rate': ['invscaling', 'adaptive'],# Las mejores estrategias de descenso
        'eta0': [0.01, 0.1]                         # Tasa inicial de aprendizaje
    }
    
    # --- CÁLCULO DE ITERACIONES ---
    total_combinaciones = len(ParameterGrid(param_grid))
    pliegues_cv = 5
    total_entrenamientos = total_combinaciones * pliegues_cv
    
    print("="*50)
    print("Resumen (SGD):")
    print(f" - Combinaciones únicas: {total_combinaciones}")
    print(f" - Pliegues de CV: {pliegues_cv}")
    print(f" - TOTAL DE ENTRENAMIENTOS A EJECUTAR: {total_entrenamientos}")
    print("="*50)
    
    # Configuramos GridSearchCV
    grid_search_sgd = GridSearchCV(
        estimator=sgd_base, 
        param_grid=param_grid, 
        cv=pliegues_cv, 
        scoring='neg_root_mean_squared_error',
        return_train_score=True,
        n_jobs=-1,
        verbose=3 # Para ver los logs de cada iteración en pantalla
    )
    
    start_time = time.time()
    
    # Entrenar la cuadrícula
    grid_search_sgd.fit(X_train, y_train)
    
    end_time = time.time()
    tiempo_total = end_time - start_time
    
    print("\n" + "="*50)
    print("SGD COMPLETADO")
    print(f"Tiempo de ejecución: {tiempo_total:.2f} segundos")
    print("="*50)
    
    # Resultados
    print("\nMejores Hiperparámetros Encontrados (SGD):")
    for param, value in grid_search_sgd.best_params_.items():
        print(f" - {param}: {value}")
        
    mejor_sgd = grid_search_sgd.best_estimator_
    rmse_cv_sgd = -grid_search_sgd.best_score_
    
    print(f"\nRMSE (Validación Cruzada 5-Fold) del SGD Ganador: {rmse_cv_sgd:.4f}")
    os.makedirs('../models', exist_ok=True)
    joblib.dump(mejor_sgd, '../models/best_sgd_model.pkl')
    return mejor_sgd, grid_search_sgd

# EJECUCIÓN 
mejor_sgd_model, info_grid_sgd = optimized_sgd_tuning(X_train_prepared, y_train)

Iniciando Búsqueda Optimizada para SGDRegressor...
Resumen (SGD):
 - Combinaciones únicas: 48
 - Pliegues de CV: 5
 - TOTAL DE ENTRENAMIENTOS A EJECUTAR: 240
Fitting 5 folds for each of 48 candidates, totalling 240 fits

SGD COMPLETADO
Tiempo de ejecución: 1.33 segundos

Mejores Hiperparámetros Encontrados (SGD):
 - alpha: 0.001
 - eta0: 0.01
 - learning_rate: adaptive
 - loss: squared_error
 - penalty: l2

RMSE (Validación Cruzada 5-Fold) del SGD Ganador: 57976.5718


In [16]:
def optimized_fine_tuning(X_train: pd.DataFrame, y_train: pd.Series) -> RandomForestRegressor:
    print("Iniciando Búsqueda Optimizada (Fine Tuning)...")
    
    rf_base = RandomForestRegressor(random_state=42)
    
    # MALLA DE HIPERPARÁMETROS REDUCIDA Y BASADA EN LITERATURA
    param_grid = [
        {
            # Bloque principal
            'n_estimators': [50, 100, 200],  
            'max_features': [6, 8, 10],       
            'max_depth': [None, 20, 40]          
        },
        {
            # Bloque sin bootstrap
            'bootstrap': [False],
            'n_estimators': [50, 100, 200],       
            'max_features': [4, 6, 8]           
        }
    ]
    
    total_combinaciones = sum(len(ParameterGrid(grid)) for grid in param_grid)
    pliegues_cv = 5
    total_entrenamientos = total_combinaciones * pliegues_cv
    
    print("="*50)
    print("Resumen (Random Forest):")
    print(f" - Combinaciones únicas: {total_combinaciones}")
    print(f" - TOTAL DE ENTRENAMIENTOS A EJECUTAR: {total_entrenamientos}")
    print("="*50)

    grid_search = GridSearchCV(
        estimator=rf_base, 
        param_grid=param_grid, 
        cv=pliegues_cv, 
        scoring='neg_root_mean_squared_error',
        return_train_score=True, 
        n_jobs=-1, 
        verbose=3 
    )
    
    start_time = time.time()
    grid_search.fit(X_train, y_train)
    end_time = time.time()
    
    minutos, segundos = divmod(end_time - start_time, 60)
    
    print("\n" + "="*50)
    print("Random Forest COMPLETADO")
    print(f"Tiempo de ejecución: {int(minutos)}m {segundos:.2f}s")
    print("="*50)
    
    print("\nMejores Hiperparámetros Encontrados (Random Forest):")
    for param, value in grid_search.best_params_.items():
        print(f" - {param}: {value}")
        
    mejor_modelo = grid_search.best_estimator_
    rmse_cv_ganador = -grid_search.best_score_
    print(f"\nRMSE (Validación Cruzada 5-Fold) del Random Forest Ganador: {rmse_cv_ganador:.4f}")
    
    os.makedirs('../models', exist_ok=True)
    joblib.dump(mejor_modelo, '../models/best_random_forest_model.pkl')
    
    return mejor_modelo, grid_search

# EJECUCIÓN
mejor_rf_model, info_grid = optimized_fine_tuning(X_train_prepared, y_train)

Iniciando Búsqueda Optimizada (Fine Tuning)...
Resumen (Random Forest):
 - Combinaciones únicas: 36
 - TOTAL DE ENTRENAMIENTOS A EJECUTAR: 180
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Random Forest COMPLETADO
Tiempo de ejecución: 2m 27.15s

Mejores Hiperparámetros Encontrados (Random Forest):
 - bootstrap: False
 - max_features: 4
 - n_estimators: 200

RMSE (Validación Cruzada 5-Fold) del Random Forest Ganador: 41391.2980


In [19]:
# Random Forest
rmse_rf = -info_grid.best_score_

# SGD Regressor
rmse_sgd = -info_grid_sgd.best_score_

# Compilar Resultados en un Benchmark Final
resultados_finales = [
    {
        "Modelo": "Random Forest Regressor (Afinado)",
        "RMSE CV (Validación)": rmse_rf,
        "Mejores Hiperparámetros": str(info_grid.best_params_) # Convertimos a string para que quepa en la tabla
    },
    {
        "Modelo": "SGD Regressor (Afinado)",
        "RMSE CV (Validación)": rmse_sgd,
        "Mejores Hiperparámetros": str(info_grid_sgd.best_params_)
    }
]

# Crear el DataFrame y ordenar del que tiene MENOR error al MAYOR
df_comparacion = pd.DataFrame(resultados_finales)
df_comparacion.sort_values(by="RMSE CV (Validación)", ascending=True, inplace=True)
df_comparacion.reset_index(drop=True, inplace=True)

# Identificar al ganador (la primera fila después de ordenar)
nombre_ganador = df_comparacion.iloc[0]['Modelo']

# Guardar el objeto del modelo ganador en una variable definitiva
if nombre_ganador == "Random Forest Regressor (Afinado)":
    modelo_definitivo = mejor_rf_model
else:
    modelo_definitivo = mejor_sgd_model

# Mostrar Resultados de la Batalla
print("="*70)
display(df_comparacion)

print(f"Ganado: {nombre_ganador.upper()}")
print("\nEl modelo ganador ha sido guardado en la variable 'modelo_definitivo'.")

,Modelo,RMSE CV (Validación),Mejores Hiperparámetros
0,Random Forest Regressor (Afinado),41391.298034,"{'bootstrap': False, 'max_features': 4, 'n_est..."
1,SGD Regressor (Afinado),57976.571756,"{'alpha': 0.001, 'eta0': 0.01, 'learning_rate'..."


Ganado: RANDOM FOREST REGRESSOR (AFINADO)

El modelo ganador ha sido guardado en la variable 'modelo_definitivo'.


In [ ]:
# 1. Conectar con tu módulo de preprocesamiento
# Ajusta la ruta si 'build_features.py' está en 'src' o en 'src/features'
sys.path.append(os.path.abspath("../src/features")) 
# Si tu archivo está directamente en src, usa: sys.path.append(os.path.abspath("../src"))
from build_features import preprocess_pipeline

print("--- INICIANDO BENCHMARK FINAL EN SET DE PRUEBA ---")

# 2. CARGAR DATOS Y ARTEFACTOS
test_path = '../data/interim/test_set.csv'
try:
    df_test_raw = pd.read_csv(test_path)
    print(f"Set de prueba crudo cargado: {df_test_raw.shape[0]} filas.")
except FileNotFoundError:
    print(f"Error: No se encontró el test set en {test_path}")

try:
    # IMPORTANTE: Carga el diccionario exacto que guardaste en tu Notebook 2
    # Si no lo guardaste como .pkl, tendrás que volver a correr preprocess_pipeline(train) rápido
    artefactos_guardados = joblib.load('../models/preprocessing_artifacts.pkl')
    
    # Cargar los modelos que afinamos
    rf_final = joblib.load('../models/best_random_forest_model.pkl')
    sgd_final = joblib.load('../models/best_sgd_model.pkl')
    print("Modelos y artefactos de preprocesamiento cargados exitosamente.")
except FileNotFoundError as e:
    print(f"Error cargando archivos: {e}")
    print("Asegúrate de haber guardado el diccionario de artefactos de tu Notebook 02 con joblib.")

# 3. PROCESAR SET DE PRUEBA (Sin Data Leakage)
# Extraemos y guardamos la variable objetivo antes de procesar
y_test = df_test_raw["median_house_value"].copy()

# Usamos tu función exactamente como la diseñaste
X_test_processed, _ = preprocess_pipeline(
    df=df_test_raw, 
    is_train=False, 
    artifacts=artefactos_guardados
)

# Asegurarnos de que no haya filtración del target en X_test
if "median_house_value" in X_test_processed.columns:
    X_test_processed = X_test_processed.drop("median_house_value", axis=1)

print(f"Set de prueba procesado: {X_test_processed.shape[0]} filas, {X_test_processed.shape[1]} columnas.")

# 4. Clasificación Final en el Set de Prueba
modelos_a_testear = {
    "Random Forest Regressor": rf_final,
    "SGD Regressor": sgd_final
}

resultados_test = []

for nombre, modelo in modelos_a_testear.items():
    # El modelo hace sus predicciones sobre los datos nunca antes vistos
    y_pred = modelo.predict(X_test_processed)
    
    # Calculamos el margen de error real (RMSE)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    resultados_test.append({
        "Modelo (Afinado)": nombre,
        "RMSE en Test ($)": rmse
    })

# Convertir a tabla elegante y ordenar
df_final_benchmark = pd.DataFrame(resultados_test).sort_values(by="RMSE en Test ($)")
df_final_benchmark.reset_index(drop=True, inplace=True)

display(df_final_benchmark)

# 5. Conclusión Final 
mejor_modelo_nombre = df_final_benchmark.iloc[0]["Modelo (Afinado)"]
mejor_rmse = df_final_benchmark.iloc[0]["RMSE en Test ($)"]

print(f"Ganado: {mejor_modelo_nombre.upper()}")
print(f"Con un error de generalización de: ${mejor_rmse:,.2f}")


--- INICIANDO BENCHMARK FINAL EN SET DE PRUEBA ---
Set de prueba crudo cargado: 4128 filas.
Modelos y artefactos de preprocesamiento cargados exitosamente.
Set de prueba procesado: 4128 filas, 18 columnas.


,Modelo (Afinado),RMSE en Test ($)
0,Random Forest Regressor,48255.622522
1,SGD Regressor,124131.098716


Ganado: RANDOM FOREST REGRESSOR
Con un error de generalización de: $48,255.62


: 

### Benchmark y Conclusión Final

Durante la fase inicial, evaluamos cuatro modelos base para identificar su comportamiento frente a la complejidad de los datos de vivienda.


Modelos Lineales (LinearRegression y SGDRegressor): Presentaron un subajuste (underfitting) ligero. Con un RMSE de entrenamiento y validación cruzada muy similares (aprox. 57,800), estos modelos demostraron ser demasiado simples para capturar las relaciones no lineales del mercado inmobiliario.


Modelos Basados en Árboles (DecisionTreeRegressor): Mostraron un sobreajuste (overfitting) crítico. El Árbol de Decisión obtuvo un RMSE de 0.0 en entrenamiento (memorizó los datos), pero su error en validación cruzada se disparó a 62,118, lo que lo hace inútil para datos nuevos.


Modelo de Ensamble (RandomForestRegressor): Fue el más prometedor desde el inicio. Aunque inicialmente presentaba sobreajuste (RMSE Train de 16,134 vs. CV de 43,704), su capacidad para manejar la complejidad lo convirtió en el candidato ideal para el fine-tuning.

| Algoritmo | RMSE Train | RMSE CV (Promedio) | Alerta / Diagnóstico |
|---|---|---|---|
| RandomForestRegressor | $16,134.42 | $43,704.96 | Prometedor (Requiere ajuste fino) |
| LinearRegression | $57,746.64 | $57,872.49 | Subajuste (Modelo muy simple) |
| SGDRegressor (Base) | $58,178.82 | $58,182.88 | Subajuste (Modelo muy simple) |
| DecisionTreeRegressor | $0.00 | $62,118.45 | Sobreajuste Crítico (Memorización) |


Justificación del Modelo Final
La selección final fue el Random Forest Regressor afinado mediante GridSearchCV como mejor. La razón principal es su equilibrio entre sesgo y varianza: mediante la búsqueda de hiperparámetros (ajustando max_features y n_estimators), logramos reducir el error y mejorar la capacidad de generalización del algoritmo.

Validación en el Set de Prueba (Benchmark)
Para garantizar que el modelo no sufra de los problemas detectados anteriormente, se realizó la prueba con el set de datos reservado (test):
El RMSE final de 48,255.62 es consistente con las pruebas de validación, lo que confirma que el modelo es robusto, generaliza correctamente y no presenta signos de sobreajuste o subajuste significativos. Estamos listos para predecir precios de vivienda con un margen de error confiable y científicamente validado.

| Modelo (Afinado con GridSearchCV) | RMSE en Test (Margen de Error) | Estado de Generalización |
|---|---|---|
| Random Forest Regressor | $48,255.62 | Excelente (Modelo Ganador) |
| SGD Regressor | $124,131.10 | Deficiente (Incapaz de generalizar) |
